# __MODEL_LABEL_MARKDOWN__: deployment

Select a published SQL candidate, open that exact immutable package for
review, and explicitly deploy it. The champion snapshot prevents a stale
notebook from overwriting a deployment that changed meanwhile.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "__MODEL_NAME__"  # Set to None to select by label only.
MODEL_LABEL = "__MODEL_LABEL__"
DEPLOYMENT_SLOT = "__DEPLOYMENT_SLOT__"
PACKAGE_VERSION = None  # None selects the latest published package.
DEPLOYMENT_REASON = ""


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

from pricing_pipeline.notebook import (
    connect,
    deploy_package,
    list_candidate_versions,
    load_registered_model,
    open_candidate,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
FRAME_ARTIFACT_PATH = MODEL_DIR / ".local" / "model_frame.joblib"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"


## Connect and list deployable SQL packages


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_candidate_versions(pricing, model=model, technical=True)
deployable = versions.loc[
    versions["package_status"].astype(str).str.upper().eq("PUBLISHED")
].copy()
display(
    deployable.loc[
        :,
        [
            "package_version",
            "model_version",
            "model_kind",
            "model_equivalence_sha256",
            "data_as_of_date",
            "manifest_id",
            "parent_rate_package_id",
            "current_rate_package_id",
        ],
    ]
)


## Select and review one published package


In [ ]:
if deployable.empty:
    raise LookupError("No published candidate packages were found.")
selected_package_version = (
    int(deployable.iloc[0]["package_version"])
    if PACKAGE_VERSION is None
    else int(PACKAGE_VERSION)
)
if selected_package_version not in set(
    deployable["package_version"].astype(int)
):
    raise ValueError("PACKAGE_VERSION is not in the displayed published list.")
reviewed = open_candidate(
    pricing,
    model=model,
    package_version=selected_package_version,
)
display(reviewed.technical)


## Deploy the reviewed package


In [ ]:
if not DEPLOYMENT_REASON.strip():
    raise ValueError("Describe the approval for changing the live package.")
deployment = deploy_package(
    pricing,
    package=reviewed,
    reason=DEPLOYMENT_REASON,
)
display(deployment)
